# PyRadiomics Feature Extraction — Setup & Exploration

Demonstrates PyRadiomics extraction on a single BraTS 2023 case. Runs all feature families (firstorder, GLCM, GLDM, GLRLM, GLSZM, NGTDM, shape) on the FLAIR modality masked to the WT boundary region.


In [ ]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd
from scipy.ndimage import binary_dilation, binary_erosion
from scipy.ndimage import generic_filter


import nibabel as nib

from skimage import feature
from scipy import ndimage as ndi
from skimage.feature import shape_index
from skimage.draw import disk
from skimage import measure
from skimage import feature
from scipy.ndimage import binary_dilation, binary_erosion


import matplotlib.pyplot as plt
from matplotlib import cm

from scipy.ndimage import gaussian_filter





import six

import radiomics
from radiomics import featureextractor
import SimpleITK as sitk

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = patient_id + '/' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz', '-seg..npz']
        outputloc = '../Results/Result/Vanilla_Unet/' + patient_id

    sample_filename1 = baseloc + pefix + suffixs[0]
    sample_img1_f = nib.load(sample_filename1)
    sample_img1 = np.asarray(sample_img1_f.dataobj)
    # PyRadiomics requires SimpleITK images, not numpy arrays
    sample_img1 = sitk.GetImageFromArray(sample_img1)
    sample_filename2 = baseloc + pefix + suffixs[1]
    sample_img2_f = nib.load(sample_filename2)
    sample_img2 = np.asarray(sample_img2_f.dataobj)
    # sample_img2  = np.rot90(sample_img2)

    sample_filename3 = baseloc + pefix + suffixs[2]
    sample_img3_f = nib.load(sample_filename3)
    sample_img3 = np.asarray(sample_img3_f.dataobj)
    # sample_img3  = np.rot90(sample_img3)

    sample_filename4 = baseloc + pefix + suffixs[3]
    sample_img4_f = nib.load(sample_filename4)
    sample_img4 = np.asarray(sample_img4_f.dataobj)
    # sample_img4  = np.rot90(sample_img4)

    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
#     sample_mask = sitk.GetImageFromArray(sample_mask)
    
    try:
        output_filename_mask = outputloc + suffixs[4]
        output_mask_f = nib.load(output_filename_mask)
        output_mask = np.asarray(output_mask_f.dataobj)
        
        prob_filename = outputloc + suffixs[5]
        probablity_img = np.load(prob_filename, allow_pickle=True)
        probablity_img = probablity_img['arr_0'][0]
        probablity_img = np.moveaxis(probablity_img, (0, 1, 2, 3), (0, 3, 2, 1))
    except ValueError :
        output_mask  = None
        probability_mask = None
        
    return sample_img1, sample_img2, sample_img3, sample_img4, sample_mask, output_mask, probablity_img

In [ ]:
dataset = 'Brats2023'
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]

In [ ]:
sample_img1, sample_img2, sample_img3, sample_img4, mask, output_mask, probability_mask  = read_MRI(dataset, 
                                                                                                    'BraTS-GLI-00006-000')


radius = 10
struct = np.ones((2*radius+1, 2*radius+1, 2*radius+1))


masks = preprocess_mask_labels(mask)
mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]

dilated_mask = binary_dilation(mask_WT,struct)
eroded_mask = binary_erosion(mask_WT,struct)
# XOR gives the boundary shell between the dilated and eroded masks
boundary_region = dilated_mask ^ eroded_mask
boundary_region = boundary_region.astype('float')

sample_mask = sample_img2.astype('bool').astype('float')
unmask_WT = sample_mask - mask_WT
unmask_WT = sitk.GetImageFromArray(unmask_WT)
mask_WT = sitk.GetImageFromArray(mask_WT)
mask_WT = sitk.GetImageFromArray(boundary_region)

# output = preprocess_mask_labels(output_mask)
# output_WT, output_TC, output_ET = output[0], output[1], output[2]

In [ ]:
settings = {}
settings['distances'] = [5]
# settings['force2D'] = True
settings['label'] = 1

# Instantiate the extractor
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)  # ** 'unpacks' the dictionary in the function call
# extractor.enableImageTypeByName('Wavelet')


print('Extraction parameters:\n\t', extractor.settings)
print('Enabled filters:\n\t', extractor.enabledImagetypes)  # Still the default parameters
print('Enabled features:\n\t', extractor.enabledFeatures)  # Still the default parameters

# extractor = featureextractor.RadiomicsFeatureExtractor()

In [ ]:
result = extractor.execute(sample_img1, mask_WT)

In [ ]:
print('Result type:', type(result))  # result is returned in a Python ordered dictionary)
print('')
print('Calculated features')
for key, value in six.iteritems(result):
    print('\t', key, ':', value)

In [ ]:
len(result)